In [1]:
from bd_utils import open_bd_drive
from read_bd_chunk import read_chunk
from compare_bd_iso import read_saved_offset

In [2]:
!ls /dev/ | grep -E "src|scd|cdrom"

cdrom


python3 compare_bd_iso.py /dev/cdrom /media/qidai/MP600/t2/diskimage.iso

In [2]:
bd_drive = "/dev/cdrom"
iso_path = "/media/qidai/MP600/t2/diskimage.iso"
progress_file = "/media/qidai/MP600/t2/diskimage.iso.txt"

In [3]:
# Read saved progress offset
saved_offset = read_saved_offset(iso_path)
print(f"Saved offset: {saved_offset:,} bytes ({saved_offset / (1024**2):,.2f} MB)")

Saved offset: 70,141,345,792 bytes (66,892.00 MB)


In [4]:
# Read BD drive chunk at saved offset
chunk_size = 4 * 1024 * 1024  # 4MB
bd_chunk = read_chunk(bd_drive, saved_offset, chunk_size)
print(f"BD chunk: {len(bd_chunk)} bytes")

# Read ISO chunk at saved offset
with open(iso_path, 'rb') as f:
    f.seek(saved_offset)
    iso_chunk = f.read(chunk_size)
print(f"ISO chunk: {len(iso_chunk)} bytes")

Opening drive /dev/cdrom...
Seeking to offset 70141345792 (0x1054C00000)...
Reading 4194304 bytes...
Successfully read 4194304 bytes.

Data Hex Dump (first 256 bytes):
0000  D2 0B 01 1E 47 10 11 16 03 93 AE 22 83 97 89 1D   ....G......"....
0010  B6 DA 1E 18 45 5A 0F 83 D4 14 CF 71 28 74 9F F9   ....EZ.....q(t..
0020  8A 6F A8 B6 A6 C6 DF 84 B0 8A 97 5E 15 40 BF 47   .o.........^.@.G
0030  FB CD 0B 09 06 8C 1F B2 B5 16 D1 9D 90 27 C2 85   .............'..
0040  3B 11 50 96 96 6A AD FA 45 3F 45 55 E9 93 21 31   ;.P..j..E?EU..!1
0050  6A 16 EF 38 B8 5D 93 7C EA 9D A4 D2 14 CA 5D 87   j..8.].|......].
0060  61 05 0E 7D F4 8B 95 7C 0C 60 C4 1D 3D 33 EC A5   a..}...|.`..=3..
0070  B3 5A B6 53 75 D1 5A FB E5 D1 14 1C D4 4B 78 61   .Z.Su.Z......Kxa
0080  14 B9 E2 51 71 BF 0F 8C B8 3F B7 F6 11 30 4D 9A   ...Qq....?...0M.
0090  D7 FA 0D 8F 04 41 BC DE 19 AD 84 94 C9 E1 02 59   .....A.........Y
00A0  A3 63 A0 45 D1 00 3D 41 C6 7E B3 F9 58 49 1F DB   .c.E..=A.~..XI..
00B0  F2 36 08 CE F5 8A B7 72

In [5]:
# Byte-by-byte comparison
if len(bd_chunk) != len(iso_chunk):
    print(f"WARNING: Size mismatch! BD: {len(bd_chunk)}, ISO: {len(iso_chunk)}")
else:
    differences = []
    for i, (b1, b2) in enumerate(zip(bd_chunk, iso_chunk)):
        if b1 != b2:
            differences.append((i, b1, b2))
    
    if differences:
        print(f"Found {len(differences)} differences:")
        # Show first 20 differences
        for offset, bd_byte, iso_byte in differences[:20]:
            print(f"  Offset {offset} (0x{offset:08X}): BD=0x{bd_byte:02X} ({bd_byte:3d}) vs ISO=0x{iso_byte:02X} ({iso_byte:3d})")
        if len(differences) > 20:
            print(f"  ... and {len(differences) - 20} more differences")
    else:
        print("✓ No differences found - chunks match!")

Found 2027 differences:
  Offset 1112080 (0x0010F810): BD=0xFC (252) vs ISO=0xBB (187)
  Offset 1112081 (0x0010F811): BD=0xC1 (193) vs ISO=0x09 (  9)
  Offset 1112082 (0x0010F812): BD=0x0C ( 12) vs ISO=0xB4 (180)
  Offset 1112083 (0x0010F813): BD=0x53 ( 83) vs ISO=0xAB (171)
  Offset 1112084 (0x0010F814): BD=0x4D ( 77) vs ISO=0xA9 (169)
  Offset 1112085 (0x0010F815): BD=0x41 ( 65) vs ISO=0x1A ( 26)
  Offset 1112086 (0x0010F816): BD=0xFF (255) vs ISO=0xBC (188)
  Offset 1112087 (0x0010F817): BD=0x15 ( 21) vs ISO=0x4A ( 74)
  Offset 1112088 (0x0010F818): BD=0xBB (187) vs ISO=0xC9 (201)
  Offset 1112089 (0x0010F819): BD=0x27 ( 39) vs ISO=0xCC (204)
  Offset 1112090 (0x0010F81A): BD=0xC0 (192) vs ISO=0xE2 (226)
  Offset 1112091 (0x0010F81B): BD=0x1F ( 31) vs ISO=0x70 (112)
  Offset 1112092 (0x0010F81C): BD=0x3D ( 61) vs ISO=0x08 (  8)
  Offset 1112093 (0x0010F81D): BD=0x9F (159) vs ISO=0x09 (  9)
  Offset 1112094 (0x0010F81E): BD=0x53 ( 83) vs ISO=0x94 (148)
  Offset 1112095 (0x0010F81F): 

In [ ]:
4, 5, 6, 

In [16]:
# Set manual offset for next chunk
manual_offset = saved_offset + chunk_size*10  # You can change this value
print(f"Manual offset: {manual_offset:,} bytes ({manual_offset / (1024**2):,.2f} MB)")

# Read and compare next chunk at manual_offset
bd_chunk2 = read_chunk(bd_drive, manual_offset, chunk_size)
print(f"BD chunk: {len(bd_chunk2)} bytes")

with open(iso_path, 'rb') as f:
    f.seek(manual_offset)
    iso_chunk2 = f.read(chunk_size)
print(f"ISO chunk: {len(iso_chunk2)} bytes")

# Byte-by-byte comparison
if len(bd_chunk2) != len(iso_chunk2):
    print(f"WARNING: Size mismatch! BD: {len(bd_chunk2)}, ISO: {len(iso_chunk2)}")
else:
    differences = []
    for i, (b1, b2) in enumerate(zip(bd_chunk2, iso_chunk2)):
        if b1 != b2:
            differences.append((i, b1, b2))
    
    if differences:
        print(f"Found {len(differences)} differences:")
        for offset, bd_byte, iso_byte in differences[:20]:
            print(f"  Offset {offset} (0x{offset+manual_offset:08X}): BD=0x{bd_byte:02X} ({bd_byte:3d}) vs ISO=0x{iso_byte:02X} ({iso_byte:3d})")
        if len(differences) > 20:
            print(f"  ... and {len(differences) - 20} more differences")
    else:
        print("✓ No differences found - chunks match!")

Manual offset: 70,183,288,832 bytes (66,932.00 MB)
Opening drive /dev/cdrom...
Seeking to offset 70183288832 (0x1057400000)...
Reading 4194304 bytes...
Successfully read 4194304 bytes.

Data Hex Dump (first 256 bytes):
0000  B9 C5 95 8B D2 4F B2 68 BD 3E F8 D6 0E F1 E2 DA   .....O.h.>......
0010  9D DA D1 39 D1 06 34 99 BE DA 59 1A 92 69 33 90   ...9..4...Y..i3.
0020  F2 60 D8 86 88 04 D5 0D 37 9B 2F 14 5F AE E7 74   .`......7./._..t
0030  5F CA F1 FB 05 DD A2 C3 D1 0D D0 89 2B 4C C2 F9   _...........+L..
0040  E7 08 AE A8 6E CC 03 68 28 08 08 0F 8B FB 26 14   ....n..h(.....&.
0050  78 92 7C D6 87 2E F5 EC FC 4F 1A B1 77 0D CB 42   x.|......O..w..B
0060  30 E9 27 03 BC 8F 46 6D 2E CC 0B 65 92 FF 60 17   0.'...Fm...e..`.
0070  4C 3D 7D D4 A2 91 F0 EC FF 04 82 28 8D B2 61 9C   L=}........(..a.
0080  63 D0 D4 A8 6A C4 59 CE D7 13 9A C5 A5 15 60 FE   c...j.Y.......`.
0090  66 49 AF 78 30 CB 5C 40 D2 FB 03 70 F0 C8 DC E0   fI.x0.\@...p....
00A0  F3 3A 03 B8 1F 87 2D AC 96 C6 2A 35 FD 93 13 